In [ ]:
# Import libraries
import requests
from dotenv import load_dotenv
import os
import pandas as pd

In [ ]:
# Loading .env file
# .env file is contains the API key and is stored in the folder
load_dotenv() # loads variables from .env file into environment
API_KEY = os.getenv("MAILERLITE_API_KEY") # retrieves API key from environment

In [ ]:
# Header guide: https://python-requests.org/python-requests-headers/
# Set up headers, needed for api requiring authentication, not public APIs
headers = {
    "Authorization": f"Bearer {API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

base_url = "https://connect.mailerlite.com/api"

MailerLite paginates the subscriber list using a cursor. Each response includes a `next_cursor` value in `meta` that tells the position of the last record returned. The loop passes this cursor back on the next request to continue from where the previous page ended. When `next_cursor` is `None`, the last page has been reached and the loop breaks. Group membership is requested via the `include=groups` parameter to preserve each subscriber's group assignments in the raw export, which carries label information used later in the pipeline.

In [ ]:
all_active_sub = []     # empty list for subscribers
cursor = None   # set cursor to none for first loop

# Writing while loop to fetch and add all subscribers into a list for EDA
while True:
    
    # Writing request to fetch subscribers
    response = requests.get(
        url=f"{base_url}/subscribers",
        headers=headers,
        params={"filter[status]" : 'active', "limit" : 100, "cursor" : cursor, "include" : "groups"}
    )
    
    data = response.json()     # outputs a dictionary, 3 keys: "data", "link", "meta"
    all_active_sub.extend(data["data"])     # adding only the "data" key, not the other two
    print(f"Fetched {len(all_active_sub)} subscribers so far...") # show progress

    cursor = data["meta"]["next_cursor"]    # calling next_cursor key within "meta" key within data dictionary
    if cursor is None:
        break

print(f"Total Active subscribers: {len(all_active_sub)}")

In [ ]:
# Converting the subscriber list to DataFrame
df = pd.DataFrame(all_active_sub)
df.info()
df.head()

In [ ]:
# Saving the DataFrame to CSV to avoid requesting API every time
df.to_csv("active_subscriber_raw.csv", index=False)